In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy samplomatic
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

<Accordion>
<AccordionItem title="Mga bersyon ng package">

Ang code sa pahinang ito ay ginawa gamit ang mga sumusunod na requirements.
Inirerekomenda naming gamitin ang mga bersyong ito o mas bago pa.

```
qiskit[all]~=2.4.1
qiskit-ibm-runtime~=0.47.0
samplomatic~=0.18.0
```
</AccordionItem>
</Accordion>
Ang mga teknik sa error mitigation na [PEA](/guides/error-mitigation-and-suppression-techniques#pea) at [PEC](/guides/error-mitigation-and-suppression-techniques#pec) ay parehong gumagamit ng noise learning component na batay sa isang [Pauli-Lindblad noise model](https://arxiv.org/abs/2201.09866), na karaniwang pinamamahalaan sa panahon ng pagpapatupad pagkatapos mag-submit ng isa o higit pang mga trabaho sa pamamagitan ng `qiskit-ibm-runtime` nang walang lokal na access sa fitted noise model. Gayunpaman, mula sa `qiskit-ibm-runtime` v0.27.1, isang [`NoiseLearner`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/noise-learner) at kaugnay na [`NoiseLearnerOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-noise-learner-options) na klase ang nilikha para makuha ang mga resulta ng mga noise learning experiment na ito. Ang mga resultang ito ay maaaring i-store nang lokal bilang isang `NoiseLearnerResult` at gamitin bilang input sa mga susunod na eksperimento. Ibinibigay ng pahinang ito ang pangkalahatang-ideya ng paggamit nito at ng mga kaugnay na opsyon na makukuha.

Bukod dito, simula sa `qiskit-ibm-runtime` v0.47.0, may bagong klase na `NoiseLearnerV3` na katugma sa Executor primitive. Ang bagong bersyong ito, na bahagi rin ng [directed execution model](/guides/directed-execution-model), ay nagbibigay sa iyo ng kakayahang tahasang tukuyin ang mga layer na nais mong matutunan.

> **Note:** `NoiseLearner` ay gumagana lamang sa EstimatorV2 at `NoiseLearnerV3` ay gumagana lamang sa Executor.

## `NoiseLearner`

### Pangkalahatang-ideya

Ang `NoiseLearner` na klase ay nagsasagawa ng mga eksperimento na nagtatasa ng mga proseso ng ingay batay sa isang Pauli-Lindblad noise model para sa isa (o higit pa) na mga circuit. Mayroon itong `run()` na pamamaraan na nagpapatakbo ng mga learning experiment at tumatanggap bilang input ng alinman sa isang listahan ng mga circuit o isang [PUB](/guides/primitive-input-output), at nagbabalik ng `NoiseLearnerResult` na naglalaman ng mga natutunang noise channel at metadata tungkol sa mga isinumiteng trabaho. Sa ibaba ay isang code snippet na nagpapakita ng paggamit ng helper program.

In [1]:
from qiskit import QuantumCircuit
from qiskit.transpiler import CouplingMap
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2
from qiskit_ibm_runtime.noise_learner import NoiseLearner
from qiskit_ibm_runtime.options import (
    NoiseLearnerOptions,
    ResilienceOptionsV2,
    EstimatorOptions,
)

# Build a circuit with two entangling layers
num_qubits = 27
edges = list(CouplingMap.from_line(num_qubits, bidirectional=False))
even_edges = edges[::2]
odd_edges = edges[1::2]

circuit = QuantumCircuit(num_qubits)
for pair in even_edges:
    circuit.cx(pair[0], pair[1])
for pair in odd_edges:
    circuit.cx(pair[0], pair[1])

# Choose a backend to run on
service = QiskitRuntimeService()
backend = service.least_busy()

# Transpile the circuit for execution
pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
circuit_to_learn = pm.run(circuit)

# Instantiate a NoiseLearner object and execute the noise learning program
learner = NoiseLearner(mode=backend)
job = learner.run([circuit_to_learn])
noise_model = job.result()

Ang resultang `NoiseLearnerResult.data` ay isang listahan ng mga [`LayerError`](../api/qiskit-ibm-runtime/utils-noise-learner-result-layer-error) na object na naglalaman ng [noise model](https://arxiv.org/abs/2201.09866) para sa bawat indibidwal na entangling layer na kabilang sa target na circuit(s). Bawat `LayerError` ay nag-iimbak ng impormasyon ng layer, sa anyo ng isang circuit at isang set ng mga qubit label, kasama ang [`PauliLindbladError`](../api/qiskit-ibm-runtime/utils-noise-learner-result-pauli-lindblad-error) para sa noise model na natuto para sa ibinigay na layer.

In [ ]:
import numpy

print(
    f"Noise learner result contains {len(noise_model.data)} entries"
    f" and has the following type:\n {type(noise_model)}\n"
)
print(
    f"Each element of `NoiseLearnerResult` then contains"
    f" an object of type:\n {type(noise_model.data[0])}\n"
)
# Results are truncated
with numpy.printoptions(threshold=200):
    print(
        f"And each of these `LayerError` objects possess"
        f" data on the generators for the error channel: \n"
        f"{noise_model.data[0].error.generators}\n"
    )
# Results are truncated
with numpy.printoptions(threshold=200):
    print(
        f"Along with the error rates: \n{noise_model.data[0].error.rates}\n"
    )

Noise learner result contains 2 entries and has the following type:
 <class 'qiskit_ibm_runtime.utils.noise_learner_result.NoiseLearnerResult'>

Each element of `NoiseLearnerResult` then contains an object of type:
 <class 'qiskit_ibm_runtime.utils.noise_learner_result.LayerError'>

And each of these `LayerError` objects possess data on the generators for the error channel: 
['IIIIIIIIIIIIIIIIIIIIIIIIIIX', 'IIIIIIIIIIIIIIIIIIIIIIIIIIY',
 'IIIIIIIIIIIIIIIIIIIIIIIIIIZ', 'IIIIIIIIIIIIIIIIIIIIIIIIIXI',
 'IIIIIIIIIIIIIIIIIIIIIIIIIXX', 'IIIIIIIIIIIIIIIIIIIIIIIIIXY',
 'IIIIIIIIIIIIIIIIIIIIIIIIIXZ', 'IIIIIIIIIIIIIIIIIIIIIIIIIYI',
 'IIIIIIIIIIIIIIIIIIIIIIIIIYX', 'IIIIIIIIIIIIIIIIIIIIIIIIIYY',
 'IIIIIIIIIIIIIIIIIIIIIIIIIYZ', 'IIIIIIIIIIIIIIIIIIIIIIIIIZI',
 'IIIIIIIIIIIIIIIIIIIIIIIIIZX', 'IIIIIIIIIIIIIIIIIIIIIIIIIZY',
 'IIIIIIIIIIIIIIIIIIIIIIIIIZZ', 'IIIIIIIIIIIIIIIIIIIIIIIIXII',
 'IIIIIIIIIIIIIIIIIIIIIIIIXIX', 'IIIIIIIIIIIIIIIIIIIIIIIIXIY',
 'IIIIIIIIIIIIIIIIIIIIIIIIXIZ', 'IIIIIIIIIIIIIIIIIIIIII

The `LayerError.error` attribute of the noise learning result contains the generators and error rates of the fitted Pauli Lindblad model, which has the form

$$ \Lambda(\rho) = \exp{\sum_j r_j \left(P_j \rho P_j^\dagger - \rho\right)}, $$

where the $r_j$ are the `LayerError.rates` and $P_j$ are the Pauli operators specified in `LayerError.generators`.

### Noise learning options

You can choose among several options to input when you instantiate a `NoiseLearner` object. These options are encapsulated by the `qiskit_ibm_runtime.options.NoiseLearnerOptions` class and include the ability to specify the maximum layers to learn, number of randomizations, and the twirling strategy, among others. Refer to the [`NoiseLearnerOptions`](/docs/api/qiskit-ibm-runtime/options-noise-learner-options) API documentation for detailed information.

Following is a simple example that shows how to use the `NoiseLearnerOptions` in a `NoiseLearner` experiment:

In [3]:
# Build a GHZ circuit
circuit = QuantumCircuit(10)
circuit.h(0)
circuit.cx(range(0, 9), range(1, 10))
# Choose a backend to run on
service = QiskitRuntimeService()
backend = service.least_busy()

# Transpile the circuit for execution
pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
circuit_to_run = pm.run(circuit_to_learn)

# Instantiate a NoiseLearnerOptions object
learner_options = NoiseLearnerOptions(
    max_layers_to_learn=3, num_randomizations=32, twirling_strategy="all"
)

# Instantiate a NoiseLearner object and execute the noise learning program
learner = NoiseLearner(mode=backend, options=learner_options)
job = learner.run([circuit_to_run])
noise_model = job.result()

Ang `LayerError.error` na katangian ng noise learning result ay naglalaman ng mga generator at error rate ng fitted na Pauli Lindblad model, na may ganitong anyo

$$ \Lambda(\rho) = \exp{\sum_j r_j \left(P_j \rho P_j^\dagger - \rho\right)}, $$

kung saan ang $r_j$ ay ang `LayerError.rates` at ang $P_j$ ay ang mga Pauli operator na tinukoy sa `LayerError.generators`.
### Mga opsyon sa noise learning
Maaari kang pumili mula sa ilang opsyon na ilalagay kapag nag-instantiate ka ng `NoiseLearner` na object. Ang mga opsyong ito ay naka-encapsulate ng `qiskit_ibm_runtime.options.NoiseLearnerOptions` na klase at kasama ang kakayahang tukuyin ang maximum na mga layer na matututunan, bilang ng mga randomization, at ang twirling strategy, bukod sa iba pa. Sumangguni sa dokumentasyon ng API ng [`NoiseLearnerOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-noise-learner-options) para sa detalyadong impormasyon.

Ang sumusunod ay isang simpleng halimbawa ng kung paano gamitin ang `NoiseLearnerOptions` sa isang `NoiseLearner` na eksperimento:

In [4]:
# Pass the noise model to the `estimator.options` attribute directly
estimator = EstimatorV2(mode=backend)
estimator.options.resilience.layer_noise_model = noise_model

In [5]:
# Specify options through a ResilienceOptionsV2 object
resilience_options = ResilienceOptionsV2(layer_noise_model=noise_model)
estimator_options = EstimatorOptions(resilience=resilience_options)
estimator = EstimatorV2(mode=backend, options=estimator_options)

In [6]:
# Specify options by using a dictionary
options_dict = {
    "resilience_level": 2,
    "resilience": {"layer_noise_model": noise_model},
}

estimator = EstimatorV2(mode=backend, options=options_dict)

After the noise model is passed into the `EstimatorV2` object, it can be used to run workloads and perform error mitigation as normal.

## NoiseLearnerV3

### Overview

Similar to `NoiseLearner`, the `NoiseLearnerV3` class performs experiments that characterize noise processes based on a Pauli-Lindblad noise model for one or more circuits. Its `run()` method takes a list of instructions, each of which must be a twirled-annotated [`BoxOp`](/docs/api/qiskit/qiskit.circuit.BoxOp) containing [ISA](/docs/guides/transpile#instruction-set-architecture) operations.


The result of a `NoiseLearnerV3` job contains a list of [`NoiseLearnerV3Result`](/docs/api/qiskit-ibm-runtime/results-noise-learner-v3-result) objects, one for each input instruction.
The following code shows how to use the helper program.

In [7]:
from qiskit import QuantumCircuit
from qiskit.transpiler import CouplingMap
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, Executor
from qiskit_ibm_runtime.noise_learner_v3 import NoiseLearnerV3
from samplomatic.transpiler import generate_boxing_pass_manager
from samplomatic.utils import find_unique_box_instructions


# Build a circuit with two entangling layers
num_qubits = 27
edges = list(CouplingMap.from_line(num_qubits, bidirectional=False))
even_edges = edges[::2]
odd_edges = edges[1::2]

circuit = QuantumCircuit(num_qubits)
for pair in even_edges:
    circuit.cx(pair[0], pair[1])
for pair in odd_edges:
    circuit.cx(pair[0], pair[1])

# Choose a backend to run on
service = QiskitRuntimeService()
backend = service.least_busy()

# Transpile the circuit for execution
pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
isa_circuit = pm.run(circuit)

# Run the boxing pass manager to group instructions into annotated boxes
boxing_pm = generate_boxing_pass_manager(
    enable_gates=True,
    enable_measures=False,
    inject_noise_targets="gates",  # no measurement mitigation
    inject_noise_strategy="uniform_modification",
)
boxed_circuit = boxing_pm.run(isa_circuit)

# Find unique boxed instructions
unique_box_instructions = find_unique_box_instructions(boxed_circuit.data)
print(f"Found {len(unique_box_instructions)} unique layers")
print(
    f"Each instruction is of type {type(unique_box_instructions[0].operation)}"
)
print(
    f"And has annotations: {unique_box_instructions[0].operation.annotations}"
)

# Instantiate a NoiseLearnerV3 object and execute the noise learning program
learner = NoiseLearnerV3(backend)
learner.options.shots_per_randomization = 128
learner.options.num_randomizations = 32
learner_job = learner.run(unique_box_instructions)
learner_result = learner_job.result()

Found 3 unique layers
Each instruction is of type <class 'qiskit.circuit.controlflow.box.BoxOp'>
And has annotations: [Twirl(group='pauli', dressing='left', decomposition='rzsx'), InjectNoise(ref='r789B', modifier_ref='', site='before')]


Pagkatapos mailagay ang noise model sa `EstimatorV2` na object, maaari na itong gamitin para magpatakbo ng mga workload at magsagawa ng error mitigation gaya ng karaniwan.
## NoiseLearnerV3

### Pangkalahatang-ideya
Katulad ng `NoiseLearner`, ang `NoiseLearnerV3` na klase ay nagsasagawa ng mga eksperimento na nagtatasa ng mga proseso ng ingay batay sa isang Pauli-Lindblad noise model para sa isa o higit pang mga circuit. Ang `run()` na pamamaraan nito ay tumatanggap ng listahan ng mga instruction, na bawat isa ay dapat na twirled-annotated [`BoxOp`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.BoxOp) na naglalaman ng mga operasyon ng [ISA](/guides/transpile#instruction-set-architecture).

Ang resulta ng isang `NoiseLearnerV3` na trabaho ay naglalaman ng listahan ng mga [`NoiseLearnerV3Result`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/results-noise-learner-v3-result) na object, isa para sa bawat input na instruction.
Ipinapakita ng sumusunod na code kung paano gamitin ang helper program.

In [ ]:
print(
    f"The Noise learner V3 result contains {len(learner_result)} entries"
    f" and each has the following type:\n {type(learner_result[0])}\n"
)
noise_map = learner_result[0].to_pauli_lindblad_map()
print(
    f"After converting to PauliLindbladMap, you can extract data "
    f" on the generators for the error channel "
    f"(truncated to 3): \n{noise_map.generators()[:3]}\n"
)
with numpy.printoptions(threshold=20):
    print(
        f"Along with the error rates "
        f"(truncated to 3): \n{noise_map.rates[:3]}\n"
    )

The Noise learner V3 result contains 3 entries and each has the following type:
 <class 'qiskit_ibm_runtime.results.noise_learner_v3.NoiseLearnerV3Result'>

After converting to PauliLindbladMap, you can extract data  on the generators for the error channel (truncated to 3): 
<QubitSparsePauliList with 3 elements on 27 qubits: [X_0, Y_0, Z_0]>

Along with the error rates (truncated to 3): 
[0.00026 0.00032 0.00023]



### Noise learning options

`NoiseLearnerV3` supports several options, including the number of randomizations and layer pair depth, among others. Similar to the primitives, you can specify the options during or after instantiating the `NoiseLearnerV3` object. The previous code example  demonstrated how to set the  `shots_per_randomization` and `num_randomizations` options. Refer to the [`NoiseLearnerV3Options`](/docs/api/qiskit-ibm-runtime/options-models-noise-learner-v3-options) API documentation for detailed information.

### Input a noise model to Executor

Executor follows the design intents specified in circuit annotations (in the form of a samplex) and options. `InjectNoise` is the annotation for specifying where to inject noise, and the `pauli_lindblad_maps` samplex argument specifies which noise map to use.

The circuit in the previous example runs through the boxing pass manager, which groups instructions into annotated boxes. The relevant code is added here for ease of understanding.
- `inject_noise_targets=”gates”` specifies to add the `InjectNoise` annotations to boxes that contain entanglers.
- `inject_noise_strategy="uniform_modification"` specifies to assign the same `ref` and `modifier_ref` to all equivalent boxes with `InjectNoise` annotations.
      - `InjectNoise.ref` is a unique identifier used to assign a noise model to that box.
      - `InjectNoise.modifier_ref` allows scaling the noise model assigned to a box by multiplicative factors.

```python
boxing_pm = generate_boxing_pass_manager(
    enable_gates=True,
    enable_measures=False,
    inject_noise_targets="gates",  # no measurement mitigation
    inject_noise_strategy="uniform_modification",
)
```

The circuit from the previous example contains three boxes, two of which contain `InjectNoise` annotations with different `ref` attributes (since they are not equivalent).

In [ ]:
# box_circuit comes from the example above
for idx, instruction in enumerate(boxed_circuit):
    # The `InjectNoise` annotation defines which boxes to inject noise.
    print(f"Annotations of box #{idx}: {instruction.operation.annotations}\n")

Annotations of box #0: [Twirl(group='pauli', dressing='left', decomposition='rzsx'), InjectNoise(ref='r789B', modifier_ref='r789B', site='before')]

Annotations of box #1: [Twirl(group='pauli', dressing='left', decomposition='rzsx'), InjectNoise(ref='r054B', modifier_ref='r054B', site='before')]

Annotations of box #2: [Twirl(group='pauli', dressing='right', decomposition='rzsx')]



Ang resulta ng trabaho ay isang listahan ng mga `NoiseLearnerV3Result` na object, isa para sa bawat input na boxed set ng mga instruction. Ang `NoiseLearnerV3Result` ay may `to_pauli_lindblad_map()` na pamamaraan na nagbabalik ng isang [`PauliLindbladMap`](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.PauliLindbladMap) na object, na may mga pamamaraan para kunin ang mga generator, error rate, at iba pa.

In [10]:
from qiskit_ibm_runtime.quantum_program import QuantumProgram
from samplomatic import build

# Generate a quantum program
program = QuantumProgram(shots=1000)

# Build the template circuit and samplex pair
template_circuit, samplex = build(boxed_circuit)

# Convert the NoiseLearnerV3 result to a dictionary
noise_maps = learner_result.to_dict(
    instructions=unique_box_instructions, require_refs=False
)

# Append the samplex item and execute
program.append_samplex_item(
    template_circuit,
    samplex=samplex,
    samplex_arguments={
        "pauli_lindblad_maps": noise_maps,
    },
)

executor = Executor(backend)
executor_job = executor.run(program)

## Next steps

<Admonition type="tip" title="Recommendations">
    - Review the [EstimatorOptions API reference](/docs/api/qiskit-ibm-runtime/options-estimator-options) and [ResilienceOptionsV2 API reference](/docs/api/qiskit-ibm-runtime/options-resilience-options-v2).
    - Learn more about [Error mitigation and suppression techniques](error-mitigation-and-suppression-techniques) that are available through Qiskit Runtime.
    - Learn how to implement [Estimator noise management](/docs/guides/estimator-noise-management).
    - Read [Migrate to V2 primitives](/docs/guides/v2-primitives).
</Admonition>